This script is to collect data from Power BI exported files 
 
Data to be transformet and exported in XLS view, or can be copied directly from here
 

In [17]:
from pathlib import Path
import pandas as pd

Paths  
two files are exported from PowerBI,   
Structure list is created by You - use line schematics to fill all the structure names you need in the project 

In [18]:
in_dir = Path(r'C:\Users\Igor.Bertyaev.APD\OneDrive - APD\_IGOR\_python\PowerBI_parse\data_in')  # working directory
asset_xls = in_dir / 'Lines - Asset Attributes (Export to Excel).xlsx'
CA_xls = in_dir / 'Condition Assessment Overview.xlsx'
str_list_xls = in_dir / 'Structure_list.xlsx'
out_dir = Path(r'C:\Users\Igor.Bertyaev.APD\OneDrive - APD\_IGOR\_python\PowerBI_parse\data_out')  # output directory
final_xls = out_dir / 'PowerBI_Extract.xlsx'

# initial names
line_name = 'OTA-WKM-C'
circuit_1 = 'OHW-OTA-1'
circuit_2 = 'OHW-OTA-2'

Now we will use our structure list xls to create pandas table  
so, creating a new tab

In [19]:
# Read all sheets from str_list_xls to get column structures
str_sheets = pd.read_excel(str_list_xls, sheet_name=None)

# Create empty DataFrames for each tab with the same columns
str_dfs = {name: pd.DataFrame(columns=df.columns) for name, df in str_sheets.items()}

# Use the 'str_list' sheet as str_list_df
str_list_df = str_dfs.get('str_list', pd.DataFrame())

# Display the table
str_list_df

,structure_id,function,contract,type,att_type_phase,att_type_ew,BE,Leg_A,Leg_B,Leg_C,Leg_D,Strengthening


In [20]:
# Read str_list_df from Excel to get the structure list
str_list_df = pd.read_excel(str_list_xls, sheet_name='str_list')

print("str_list_df columns:", str_list_df.columns.tolist())

# Fill other columns from asset_xls Tower tab

# Mapping from Excel headers to DataFrame column names
column_mapping = {
    'Device Position': 'structure_id',
    'Tower Contract': 'contract',
    'Tower Type': 'type',
    'Insulator Attach Type': 'att_type_phase',
    'Earthwire Attach Type': 'att_type_ew',
    'Body Ext': 'BE',
    'Leg A Length': 'Leg_A',
    'Leg B Length': 'Leg_B',
    'Leg C Length': 'Leg_C',
    'Leg D Length': 'Leg_D',
    'Twr Strengthened Y/N': 'Strengthening'
    # Removed 'Circuit': 'circuit'
}

tower_df = pd.read_excel(asset_xls, sheet_name='Tower', usecols=list(column_mapping.keys()))
tower_df.rename(columns=column_mapping, inplace=True)

print("Tower df columns after rename:", tower_df.columns.tolist())

# Group by structure_id and aggregate
def aggregate_func(series):
    unique_vals = series.dropna().unique()
    if len(unique_vals) == 1:
        return unique_vals[0]
    else:
        return ' / '.join(map(str, unique_vals))

aggregated_df = tower_df.groupby('structure_id').agg(aggregate_func).reset_index()

print("Aggregated df columns:", aggregated_df.columns.tolist())

# Merge aggregated data with str_list_df
str_list_df = str_list_df.merge(aggregated_df, on='structure_id', how='left')

# Clean up overlapping columns by using the merged values
overlapping_cols = ['contract', 'type', 'att_type_phase', 'att_type_ew', 'BE', 'Leg_A', 'Leg_B', 'Leg_C', 'Leg_D', 'Strengthening']
for col in overlapping_cols:
    if col + '_y' in str_list_df.columns:
        str_list_df[col] = str_list_df[col + '_y']
        str_list_df.drop(columns=[col + '_x', col + '_y'], inplace=True, errors='ignore')

# Display updated table
str_list_df

str_list_df columns: ['structure_id', 'function', 'contract', 'type', 'att_type_phase', 'att_type_ew', 'BE', 'Leg_A', 'Leg_B', 'Leg_C', 'Leg_D', 'Strengthening']
Tower df columns after rename: ['structure_id', 'BE', 'att_type_ew', 'att_type_phase', 'Leg_A', 'Leg_B', 'Leg_C', 'Leg_D', 'contract', 'type', 'Strengthening']
Aggregated df columns: ['structure_id', 'BE', 'att_type_ew', 'att_type_phase', 'Leg_A', 'Leg_B', 'Leg_C', 'Leg_D', 'contract', 'type', 'Strengthening']


,structure_id,function,contract,type,att_type_phase,att_type_ew,BE,Leg_A,Leg_B,Leg_C,Leg_D,Strengthening
0,OTA-WKM-C0400,NaN,C385B,E,HSH,PLT,5.0,3.04,3.04,3.04,3.04,N
1,OTA-WKM-C0401,NaN,C385B,C,HBK,PLT,,6.08,6.08,6.08,6.08,N
2,OTA-WKM-C0402,NaN,C385B,B,HSH,PLT,,3.04,1.52,3.04,3.04,N
3,OTA-WKM-C0403,NaN,C385B,C,HBK,PLT,,3.04,3.04,3.04,3.04,N
4,OTA-WKM-C0404,NaN,C385B,A,HSH,PLT,,3.04,1.52,3.04,3.04,N
...,...,...,...,...,...,...,...,...,...,...,...,...
87,OTA-WKM-C0487,NaN,C385B,A,PLT,PLT,3.0,6.08,6.08,6.08,6.08,N
88,OTA-WKM-C0488,NaN,C385B,E,PLT,PLT,,5.00,5.00,5.00,5.00,N
89,OTA-WKM-C0489,NaN,C385B,A,HSH,PLT,5.0,6.08,6.08,6.08,6.08,N
90,OTA-WKM-C0490,NaN,C385B,A,HSH,PLT,5.0,6.08,6.08,6.08,6.08,N


For future to update:
- Use not only Tower tab, but also Pole and Termination, as some structures may be placed there

Add columns from assets: 
- foundation - OK
- insulators - OK
- EW_assembly - OK
- Replace cirquit names with CCT1, CCT2 - OK
  
Create column 'function' based on insulators - OK


1. Add Foundation

In [21]:
# Add foundation column from 'Tower Foundation' tab
foundation_df = pd.read_excel(asset_xls, sheet_name='Tower Foundation', usecols=['Device Position', 'Foundation Type'])
foundation_df.rename(columns={'Device Position': 'structure_id', 'Foundation Type': 'foundation'}, inplace=True)

# Aggregate foundation if needed
def aggregate_func(series):
    unique_vals = series.dropna().unique()
    if len(unique_vals) == 1:
        return unique_vals[0]
    else:
        return ' / '.join(map(str, unique_vals))

foundation_agg = foundation_df.groupby('structure_id').agg({'foundation': aggregate_func}).reset_index()

# Replace existing foundation column with the new aggregated one
str_list_df.drop(columns=['foundation'], inplace=True, errors='ignore')
str_list_df = str_list_df.merge(foundation_agg[['structure_id', 'foundation']], on='structure_id', how='left')

# Display updated table
str_list_df

,structure_id,function,contract,type,att_type_phase,att_type_ew,BE,Leg_A,Leg_B,Leg_C,Leg_D,Strengthening,foundation
0,OTA-WKM-C0400,NaN,C385B,E,HSH,PLT,5.0,3.04,3.04,3.04,3.04,N,PBP
1,OTA-WKM-C0401,NaN,C385B,C,HBK,PLT,,6.08,6.08,6.08,6.08,N,PBP
2,OTA-WKM-C0402,NaN,C385B,B,HSH,PLT,,3.04,1.52,3.04,3.04,N,COG
3,OTA-WKM-C0403,NaN,C385B,C,HBK,PLT,,3.04,3.04,3.04,3.04,N,COG
4,OTA-WKM-C0404,NaN,C385B,A,HSH,PLT,,3.04,1.52,3.04,3.04,N,GRG
...,...,...,...,...,...,...,...,...,...,...,...,...,...
87,OTA-WKM-C0487,NaN,C385B,A,PLT,PLT,3.0,6.08,6.08,6.08,6.08,N,COG
88,OTA-WKM-C0488,NaN,C385B,E,PLT,PLT,,5.00,5.00,5.00,5.00,N,GRG
89,OTA-WKM-C0489,NaN,C385B,A,HSH,PLT,5.0,6.08,6.08,6.08,6.08,N,GRG
90,OTA-WKM-C0490,NaN,C385B,A,HSH,PLT,5.0,6.08,6.08,6.08,6.08,N,PBP


2.1. Add a new DF with Insulators

In [22]:
# Create insulators DataFrame from 'Insulators & Hardware' tab
ins_df = pd.read_excel(asset_xls, sheet_name='Insulators & Hardware', usecols=[
    'Circuit', 'Device Position', 'Jumper Ins Qty', 'Jumper Std Assy', 
    'Strain Back Ins Qty', 'Strain Back Std Assy', 'Strain Fwd Ins Qty', 
    'Strain Fwd Std Assy', 'Susp Ins Qty', 'Susp Std Assy', 'Weight Qty'
], dtype={
    'Jumper Ins Qty': 'Int64',
    'Strain Back Ins Qty': 'Int64',
    'Strain Fwd Ins Qty': 'Int64',
    'Susp Ins Qty': 'Int64'
})

ins_df.rename(columns={'Device Position': 'structure_id'}, inplace=True)

# Filter to only structures in str_list_df
ins_df = ins_df[ins_df['structure_id'].isin(str_list_df['structure_id'])]


# Convert columns to object dtype to avoid dtype warnings
for col in ins_df.columns:
    if col not in ['structure_id', 'Circuit']:
        ins_df[col] = ins_df[col].astype(object)

# Prefix circuit to each value in ins_df
for idx, row in ins_df.iterrows():
    circuit = row['Circuit']
    for col in ins_df.columns:
        if col not in ['structure_id', 'Circuit'] and pd.notna(row[col]):
            ins_df.at[idx, col] = f"{circuit}: {row[col]}"

# Drop Circuit column
ins_df.drop(columns=['Circuit'], inplace=True)

# Replace circuit names in ins_df with CCT1 and CCT2
ins_df = ins_df.apply(lambda col: col.str.replace(circuit_1, "CCT1", regex=False) if col.dtype == 'object' else col)
ins_df = ins_df.apply(lambda col: col.str.replace(circuit_2, "CCT2", regex=False) if col.dtype == 'object' else col)

# Aggregate to one row per structure_id
def combine_unique(series):
    vals = series.dropna().unique()
    return ' / '.join(map(str, vals)) if len(vals) > 0 else None

ins_agg = ins_df.groupby('structure_id').agg(combine_unique).reset_index()


# Display the aggregated insulators DataFrame
ins_agg

,structure_id,Jumper Ins Qty,Jumper Std Assy,Strain Back Ins Qty,Strain Back Std Assy,Strain Fwd Ins Qty,Strain Fwd Std Assy,Susp Ins Qty,Susp Std Assy,Weight Qty
0,OTA-WKM-C0400,CCT2: 42 / CCT1: 42,CCT2: 10A / CCT1: 848C,CCT2: 78 / CCT1: 84,CCT2: 13S / CCT1: 760C,CCT2: 78 / CCT1: 84,CCT2: 13S / CCT1: 760C,None,None,None
1,OTA-WKM-C0401,None,None,None,None,None,None,CCT1: 39 / CCT2: 39,CCT1: 11CM / CCT2: 11E,None
2,OTA-WKM-C0402,None,None,None,None,None,None,CCT2: 39 / CCT1: 39,CCT2: 10B / CCT1: 10B,None
3,OTA-WKM-C0403,None,None,None,None,None,None,CCT2: 39 / CCT1: 39,CCT2: 10B / CCT1: 10B,None
4,OTA-WKM-C0404,None,None,None,None,None,None,CCT1: 42 / CCT2: 42,CCT1: 11DM / CCT2: 11E,None
...,...,...,...,...,...,...,...,...,...,...
87,OTA-WKM-C0487,None,None,None,None,None,None,CCT1: 3 / CCT2: 3,None,None
88,OTA-WKM-C0488,CCT2: 3 / CCT1: 3,CCT2: 523 / CCT1: 523,CCT2: 72 / CCT1: 72,CCT2: 13S / CCT1: 13S,CCT2: 72 / CCT1: 72,CCT2: 13S / CCT1: 13S,None,None,None
89,OTA-WKM-C0489,None,None,None,None,None,None,CCT2: 42 / CCT1: 42,CCT2: 10AS / CCT1: 10AS,None
90,OTA-WKM-C0490,None,None,None,None,None,None,CCT1: 3 / CCT2: 3,CCT1: 524 / CCT2: 524,None


2.2. Add Insulators to structure list df

In [23]:
# Add insulator columns to str_list_df
str_list_df = str_list_df.merge(ins_agg, on='structure_id', how='left')

# Fill 'function' column based on Strain columns
strain_cols = ['Strain Back Ins Qty', 'Strain Back Std Assy', 'Strain Fwd Ins Qty', 'Strain Fwd Std Assy']
str_list_df['function'] = str_list_df.apply(
    lambda row: "Strain" if any(pd.notna(row[col]) for col in strain_cols) else "Suspension",
    axis=1
)

# Display updated str_list_df
str_list_df

,structure_id,function,contract,type,att_type_phase,att_type_ew,BE,Leg_A,Leg_B,Leg_C,...,foundation,Jumper Ins Qty,Jumper Std Assy,Strain Back Ins Qty,Strain Back Std Assy,Strain Fwd Ins Qty,Strain Fwd Std Assy,Susp Ins Qty,Susp Std Assy,Weight Qty
0,OTA-WKM-C0400,Strain,C385B,E,HSH,PLT,5.0,3.04,3.04,3.04,...,PBP,CCT2: 42 / CCT1: 42,CCT2: 10A / CCT1: 848C,CCT2: 78 / CCT1: 84,CCT2: 13S / CCT1: 760C,CCT2: 78 / CCT1: 84,CCT2: 13S / CCT1: 760C,None,None,None
1,OTA-WKM-C0401,Suspension,C385B,C,HBK,PLT,,6.08,6.08,6.08,...,PBP,None,None,None,None,None,None,CCT1: 39 / CCT2: 39,CCT1: 11CM / CCT2: 11E,None
2,OTA-WKM-C0402,Suspension,C385B,B,HSH,PLT,,3.04,1.52,3.04,...,COG,None,None,None,None,None,None,CCT2: 39 / CCT1: 39,CCT2: 10B / CCT1: 10B,None
3,OTA-WKM-C0403,Suspension,C385B,C,HBK,PLT,,3.04,3.04,3.04,...,COG,None,None,None,None,None,None,CCT2: 39 / CCT1: 39,CCT2: 10B / CCT1: 10B,None
4,OTA-WKM-C0404,Suspension,C385B,A,HSH,PLT,,3.04,1.52,3.04,...,GRG,None,None,None,None,None,None,CCT1: 42 / CCT2: 42,CCT1: 11DM / CCT2: 11E,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87,OTA-WKM-C0487,Suspension,C385B,A,PLT,PLT,3.0,6.08,6.08,6.08,...,COG,None,None,None,None,None,None,CCT1: 3 / CCT2: 3,None,None
88,OTA-WKM-C0488,Strain,C385B,E,PLT,PLT,,5.00,5.00,5.00,...,GRG,CCT2: 3 / CCT1: 3,CCT2: 523 / CCT1: 523,CCT2: 72 / CCT1: 72,CCT2: 13S / CCT1: 13S,CCT2: 72 / CCT1: 72,CCT2: 13S / CCT1: 13S,None,None,None
89,OTA-WKM-C0489,Suspension,C385B,A,HSH,PLT,5.0,6.08,6.08,6.08,...,GRG,None,None,None,None,None,None,CCT2: 42 / CCT1: 42,CCT2: 10AS / CCT1: 10AS,None
90,OTA-WKM-C0490,Suspension,C385B,A,HSH,PLT,5.0,6.08,6.08,6.08,...,PBP,None,None,None,None,None,None,CCT1: 3 / CCT2: 3,CCT1: 524 / CCT2: 524,None


3. Add EW

In [24]:
# Create earthwire sets db
ew_ins_df = pd.read_excel(asset_xls, sheet_name='Earthwire HW Set (Str Eqp)', usecols=[
    'Circuit', 'Device Position', 'Armour Rod Qty', 'EWHW Assembly Qty', 
    'EWHW Assembly StdAssy'
], dtype={
    'Armour Rod Qty': 'Int64',
    'EWHW Assembly Qty': 'Int64'
})

ew_ins_df.rename(columns={'Device Position': 'structure_id'}, inplace=True)

# Filter to only structures in str_list_df
ew_ins_df = ew_ins_df[ew_ins_df['structure_id'].isin(str_list_df['structure_id'])]


aggregated_ew_ins_df = ew_ins_df.groupby('structure_id').agg(aggregate_func).reset_index()

# Merge aggregated data with str_list_df
str_list_df = str_list_df.merge(aggregated_ew_ins_df, on='structure_id', how='left')

# Display updated str_list_df
str_list_df

,structure_id,function,contract,type,att_type_phase,att_type_ew,BE,Leg_A,Leg_B,Leg_C,...,Strain Back Std Assy,Strain Fwd Ins Qty,Strain Fwd Std Assy,Susp Ins Qty,Susp Std Assy,Weight Qty,Circuit,Armour Rod Qty,EWHW Assembly Qty,EWHW Assembly StdAssy
0,OTA-WKM-C0400,Strain,C385B,E,HSH,PLT,5.0,3.04,3.04,3.04,...,CCT2: 13S / CCT1: 760C,CCT2: 78 / CCT1: 84,CCT2: 13S / CCT1: 760C,None,None,None,OHW-OTA-1 / OHW-OTA-2,0,4,22B
1,OTA-WKM-C0401,Suspension,C385B,C,HBK,PLT,,6.08,6.08,6.08,...,None,None,None,CCT1: 39 / CCT2: 39,CCT1: 11CM / CCT2: 11E,None,OHW-OTA-2 / OHW-OTA-1,2,2,27
2,OTA-WKM-C0402,Suspension,C385B,B,HSH,PLT,,3.04,1.52,3.04,...,None,None,None,CCT2: 39 / CCT1: 39,CCT2: 10B / CCT1: 10B,None,OHW-OTA-2 / OHW-OTA-1,2,2,27
3,OTA-WKM-C0403,Suspension,C385B,C,HBK,PLT,,3.04,3.04,3.04,...,None,None,None,CCT2: 39 / CCT1: 39,CCT2: 10B / CCT1: 10B,None,OHW-OTA-2 / OHW-OTA-1,2,2,27
4,OTA-WKM-C0404,Suspension,C385B,A,HSH,PLT,,3.04,1.52,3.04,...,None,None,None,CCT1: 42 / CCT2: 42,CCT1: 11DM / CCT2: 11E,None,OHW-OTA-2 / OHW-OTA-1,2,2,27
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87,OTA-WKM-C0487,Suspension,C385B,A,PLT,PLT,3.0,6.08,6.08,6.08,...,None,None,None,CCT1: 3 / CCT2: 3,None,None,OHW-OTA-1 / OHW-OTA-2,2,2,27
88,OTA-WKM-C0488,Strain,C385B,E,PLT,PLT,,5.00,5.00,5.00,...,CCT2: 13S / CCT1: 13S,CCT2: 72 / CCT1: 72,CCT2: 13S / CCT1: 13S,None,None,None,OHW-OTA-1 / OHW-OTA-2,,4,22B
89,OTA-WKM-C0489,Suspension,C385B,A,HSH,PLT,5.0,6.08,6.08,6.08,...,None,None,None,CCT2: 42 / CCT1: 42,CCT2: 10AS / CCT1: 10AS,None,OHW-OTA-2 / OHW-OTA-1,2,2,27
90,OTA-WKM-C0490,Suspension,C385B,A,HSH,PLT,5.0,6.08,6.08,6.08,...,None,None,None,CCT1: 3 / CCT2: 3,CCT1: 524 / CCT2: 524,None,OHW-OTA-1 / OHW-OTA-2,2,2,27


4. Write to Excel

In [25]:
# Export str_list_df back to str_list_xls 'str_list' sheet
from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows

# Load the workbook
wb = load_workbook(str_list_xls)

# Remove the existing 'str_list' sheet if it exists
if 'str_list_filled' in wb.sheetnames:
    wb.remove(wb['str_list_filled'])

# Create new 'str_list' sheet
ws = wb.create_sheet('str_list_filled')

# Write str_list_df to the sheet
for r in dataframe_to_rows(str_list_df, index=False, header=True):
    ws.append(r)

# Save the workbook
wb.save(str_list_xls)

print("Data exported to str_list_xls 'str_list_filled' sheet.")

Data exported to str_list_xls 'str_list_filled' sheet.


Part 2.  CA data collection
  

In [26]:
# Create CA db
ca_df = pd.read_excel(CA_xls, sheet_name='Export', usecols=[
    'Asset Description', 'Device Position', 'Meter Description', 'Measurement', 
    'Measurement Date'
], dtype={
    'Measurement': 'Int64'
}, parse_dates=['Measurement Date'])

ca_df.rename(columns={'Device Position': 'structure_id'}, inplace=True)

# Filter to only structures in str_list_df
ca_df = ca_df[ca_df['structure_id'].isin(str_list_df['structure_id'])]

ca_df

,Asset Description,structure_id,Meter Description,Measurement,Measurement Date
19325,OTA-WKM-C-0400-Tower,OTA-WKM-C0400,Body Major Steel Condition,60,2011-04-21 00:00:00
19326,OTA-WKM-C-0400-Tower,OTA-WKM-C0400,Body Minor Steel Condition,50,2011-04-21 00:00:00
19327,OTA-WKM-C-0400-Tower,OTA-WKM-C0400,Crossarm Bolt Condition,50,2011-04-21 00:00:00
19328,OTA-WKM-C-0400-Tower,OTA-WKM-C0400,Crossarm Major Steel Condition,60,2011-04-21 00:00:00
19329,OTA-WKM-C-0400-Tower,OTA-WKM-C0400,Crossarm Minor Steel Condition,50,2011-04-21 00:00:00
...,...,...,...,...,...
23912,OTA-WKM-C-0491-Insulator Attachment-OHW-OTA-1,OTA-WKM-C0491,Attachment Point Through Bolt and Nut Condition,70,2022-09-14 14:53:08
23913,OTA-WKM-C-0491-Insulator Attachment-OHW-OTA-2,OTA-WKM-C0491,Attachment Point Condition,65,2022-09-14 14:54:19
23914,OTA-WKM-C-0491-Insulator Attachment-OHW-OTA-2,OTA-WKM-C0491,Attachment Point Nuts and Bolt Condition,60,2022-09-14 14:54:19
23915,OTA-WKM-C-0491-Insulator Attachment-OHW-OTA-2,OTA-WKM-C0491,Attachment Point Steel Condition,75,2022-09-14 14:54:19


Transpose column with measurements and dates to get one row per structure

In [27]:
# trying to transpose column with meter description to get one row per structure_id and columns for each meter type with corresponding measurement values

ca_df_T = ca_df.pivot_table(index='structure_id', columns='Meter Description', values=['Measurement', 'Measurement Date'], aggfunc='first').reset_index()

ca_df_T.columns = ['_ '.join([str(i) for i in col if i]) for col in ca_df_T.columns]   # flatten multiindex columns

ca_df_T.columns = [col.replace('Measurement Date_', 'Date_') for col in ca_df_T.columns]     # rename date columns to have Date_ prefix instead of Measurement Date_

#ca_df_T


2.1. Creating Tower CA tab

In [28]:
df_Tower_CA = str_list_df.copy()    # create a copy of str_list_df to merge with CA data and then export to Excel
df_Tower_CA.columns


Index(['structure_id', 'function', 'contract', 'type', 'att_type_phase',
       'att_type_ew', 'BE', 'Leg_A', 'Leg_B', 'Leg_C', 'Leg_D',
       'Strengthening', 'foundation', 'Jumper Ins Qty', 'Jumper Std Assy',
       'Strain Back Ins Qty', 'Strain Back Std Assy', 'Strain Fwd Ins Qty',
       'Strain Fwd Std Assy', 'Susp Ins Qty', 'Susp Std Assy', 'Weight Qty',
       'Circuit', 'Armour Rod Qty', 'EWHW Assembly Qty',
       'EWHW Assembly StdAssy'],
      dtype='object')

Delete columns we do not use

In [29]:
df_Tower_CA.drop(columns=['att_type_phase', 'att_type_ew', 'foundation', 'Jumper Ins Qty', 'Jumper Std Assy',
       'Strain Back Ins Qty', 'Strain Back Std Assy', 'Strain Fwd Ins Qty',
       'Strain Fwd Std Assy', 'Susp Ins Qty', 'Susp Std Assy', 'Weight Qty',
       'Circuit', 'Armour Rod Qty', 'EWHW Assembly Qty',
       'EWHW Assembly StdAssy'], inplace=True, errors='ignore')

df_Tower_CA


,structure_id,function,contract,type,BE,Leg_A,Leg_B,Leg_C,Leg_D,Strengthening
0,OTA-WKM-C0400,Strain,C385B,E,5.0,3.04,3.04,3.04,3.04,N
1,OTA-WKM-C0401,Suspension,C385B,C,,6.08,6.08,6.08,6.08,N
2,OTA-WKM-C0402,Suspension,C385B,B,,3.04,1.52,3.04,3.04,N
3,OTA-WKM-C0403,Suspension,C385B,C,,3.04,3.04,3.04,3.04,N
4,OTA-WKM-C0404,Suspension,C385B,A,,3.04,1.52,3.04,3.04,N
...,...,...,...,...,...,...,...,...,...,...
87,OTA-WKM-C0487,Suspension,C385B,A,3.0,6.08,6.08,6.08,6.08,N
88,OTA-WKM-C0488,Strain,C385B,E,,5.00,5.00,5.00,5.00,N
89,OTA-WKM-C0489,Suspension,C385B,A,5.0,6.08,6.08,6.08,6.08,N
90,OTA-WKM-C0490,Suspension,C385B,A,5.0,6.08,6.08,6.08,6.08,N


In [30]:
# join Legs into one column

df_Tower_CA['Legs'] = df_Tower_CA.apply(lambda row: ' - '.join([str(row[col]) for col in ['Leg_A', 'Leg_B', 'Leg_C', 'Leg_D'] if pd.notna(row[col])]), axis=1)
df_Tower_CA.drop(columns=['Leg_A', 'Leg_B', 'Leg_C', 'Leg_D'], inplace=True, errors='ignore')
df_Tower_CA


,structure_id,function,contract,type,BE,Strengthening,Legs
0,OTA-WKM-C0400,Strain,C385B,E,5.0,N,3.04 - 3.04 - 3.04 - 3.04
1,OTA-WKM-C0401,Suspension,C385B,C,,N,6.08 - 6.08 - 6.08 - 6.08
2,OTA-WKM-C0402,Suspension,C385B,B,,N,3.04 - 1.52 - 3.04 - 3.04
3,OTA-WKM-C0403,Suspension,C385B,C,,N,3.04 - 3.04 - 3.04 - 3.04
4,OTA-WKM-C0404,Suspension,C385B,A,,N,3.04 - 1.52 - 3.04 - 3.04
...,...,...,...,...,...,...,...
87,OTA-WKM-C0487,Suspension,C385B,A,3.0,N,6.08 - 6.08 - 6.08 - 6.08
88,OTA-WKM-C0488,Strain,C385B,E,,N,5.0 - 5.0 - 5.0 - 5.0
89,OTA-WKM-C0489,Suspension,C385B,A,5.0,N,6.08 - 6.08 - 6.08 - 6.08
90,OTA-WKM-C0490,Suspension,C385B,A,5.0,N,6.08 - 6.08 - 6.08 - 6.08


Add CA

In [31]:
# add CA data 

df_Tower_CA_merged = df_Tower_CA.merge(ca_df_T[['structure_id', 'Measurement_ Body Major Steel Condition', 'Date_ Body Major Steel Condition', 
                                                'Measurement_ Body Minor Steel Condition', 'Date_ Body Minor Steel Condition',
                                                'Measurement_ Crossarm Major Steel Condition', 'Date_ Crossarm Major Steel Condition',
                                                'Measurement_ Crossarm Minor Steel Condition', 'Date_ Crossarm Minor Steel Condition',
                                                'Measurement_ Tower Bolt Condition', 'Date_ Tower Bolt Condition',
                                                'Measurement_ Crossarm Bolt Condition', 'Date_ Crossarm Bolt Condition'
                                                ]], on='structure_id', how='left')

# join some CA with minimum values and most recent dates
df_Tower_CA_merged['body'] = df_Tower_CA_merged[['Measurement_ Body Major Steel Condition', 'Measurement_ Body Minor Steel Condition']].min(axis=1)
df_Tower_CA_merged['xarm'] = df_Tower_CA_merged[['Measurement_ Crossarm Major Steel Condition', 'Measurement_ Crossarm Minor Steel Condition']].min(axis=1)
df_Tower_CA_merged['bolts'] = df_Tower_CA_merged[['Measurement_ Tower Bolt Condition', 'Measurement_ Crossarm Bolt Condition']].min(axis=1)
df_Tower_CA_merged['date'] = df_Tower_CA_merged[['Date_ Tower Bolt Condition', 'Date_ Crossarm Bolt Condition', 'Date_ Body Major Steel Condition', 'Date_ Body Minor Steel Condition']].max(axis=1)

df_Tower_CA_merged.drop(columns=['Measurement_ Body Major Steel Condition', 'Measurement_ Body Minor Steel Condition', 'Measurement_ Crossarm Major Steel Condition', 'Measurement_ Crossarm Minor Steel Condition', 'Measurement_ Tower Bolt Condition', 'Measurement_ Crossarm Bolt Condition'], inplace=True, errors='ignore')
df_Tower_CA_merged.drop(columns=['Date_ Body Major Steel Condition', 'Date_ Body Minor Steel Condition', 'Date_ Crossarm Major Steel Condition', 'Date_ Crossarm Minor Steel Condition', 'Date_ Tower Bolt Condition', 'Date_ Crossarm Bolt Condition'], inplace=True, errors='ignore')

df_Tower_CA_merged = df_Tower_CA_merged[['structure_id', 'contract', 'type', 'function', 'BE', 'Legs', 'body', 'xarm', 'bolts', 'date', 'Strengthening']]
df_Tower_CA_merged


,structure_id,contract,type,function,BE,Legs,body,xarm,bolts,date,Strengthening
0,OTA-WKM-C0400,C385B,E,Strain,5.0,3.04 - 3.04 - 3.04 - 3.04,50,50,50,2011-04-21 00:00:00,N
1,OTA-WKM-C0401,C385B,C,Suspension,,6.08 - 6.08 - 6.08 - 6.08,55,50,50,2009-08-18 00:00:00,N
2,OTA-WKM-C0402,C385B,B,Suspension,,3.04 - 1.52 - 3.04 - 3.04,55,55,50,2009-08-18 00:00:00,N
3,OTA-WKM-C0403,C385B,C,Suspension,,3.04 - 3.04 - 3.04 - 3.04,50,50,50,2007-08-08 00:00:00,N
4,OTA-WKM-C0404,C385B,A,Suspension,,3.04 - 1.52 - 3.04 - 3.04,55,60,50,2007-07-30 00:00:00,N
...,...,...,...,...,...,...,...,...,...,...,...
87,OTA-WKM-C0487,C385B,A,Suspension,3.0,6.08 - 6.08 - 6.08 - 6.08,65,65,65,2011-05-06 00:00:00,N
88,OTA-WKM-C0488,C385B,E,Strain,,5.0 - 5.0 - 5.0 - 5.0,70,60,65,2007-06-14 00:00:00,N
89,OTA-WKM-C0489,C385B,A,Suspension,5.0,6.08 - 6.08 - 6.08 - 6.08,70,70,60,2011-05-10 00:00:00,N
90,OTA-WKM-C0490,C385B,A,Suspension,5.0,6.08 - 6.08 - 6.08 - 6.08,60,60,60,2011-05-10 00:00:00,N


write to xls

In [34]:
df_Tower_CA_merged = df_Tower_CA_merged.fillna(0)
df_Tower_CA_merged['date'] = pd.to_datetime(df_Tower_CA_merged['date']) # convert to date format
df_Tower_CA_merged['date'] = df_Tower_CA_merged['date'].dt.strftime('%Y/%m/%d') # convert to string format for Excel

wb = load_workbook(str_list_xls)

# Remove the existing 'Tower_CA' sheet if it exists
if 'Tower_CA' in wb.sheetnames:
    wb.remove(wb['Tower_CA'])

# Create new 'Tower_CA' sheet
ws = wb.create_sheet('Tower_CA')

# Write Tower_CA dataframe to the sheet
for r in dataframe_to_rows(df_Tower_CA_merged, index=False, header=True):
    ws.append(r)

# Save the workbook
wb.save(str_list_xls)

print("Data exported to str_list_xls 'Tower_CA' sheet.")

Data exported to str_list_xls 'Tower_CA' sheet.


2.2. Foundation CA

In [35]:
# create a separate df for foundation CA data only to export to Excel
df_Found_CA = str_list_df[[ 'structure_id', 'contract', 'type', 'foundation']].copy()    # create a copy of str_list_df to merge with CA data and then export to Excel
df_Found_CA


,structure_id,contract,type,foundation
0,OTA-WKM-C0400,C385B,E,PBP
1,OTA-WKM-C0401,C385B,C,PBP
2,OTA-WKM-C0402,C385B,B,COG
3,OTA-WKM-C0403,C385B,C,COG
4,OTA-WKM-C0404,C385B,A,GRG
...,...,...,...,...
87,OTA-WKM-C0487,C385B,A,COG
88,OTA-WKM-C0488,C385B,E,GRG
89,OTA-WKM-C0489,C385B,A,GRG
90,OTA-WKM-C0490,C385B,A,PBP


In [36]:
# add CA data 
df_Found_CA_merged = df_Found_CA.merge(ca_df_T[['structure_id', 
                                                'Measurement_ Leg A Condition', 'Date_ Leg A Condition',
                                                'Measurement_ Leg B Condition', 'Date_ Leg B Condition',
                                                'Measurement_ Leg C Condition', 'Date_ Leg C Condition',
                                                'Measurement_ Leg D Condition', 'Date_ Leg D Condition',
                                                'Measurement_ Leg A Connect Condition', 'Date_ Leg A Connect Condition',
                                                'Measurement_ Leg B Connect Condition', 'Date_ Leg B Connect Condition',
                                                'Measurement_ Leg C Connect Condition', 'Date_ Leg C Connect Condition',
                                                'Measurement_ Leg D Connect Condition', 'Date_ Leg D Connect Condition',
                                                ]], on='structure_id', how='left')

# join some CA with minimum values and most recent dates
df_Found_CA_merged['date'] = df_Found_CA_merged[['Date_ Leg A Condition', 'Date_ Leg B Condition', 'Date_ Leg C Condition', 'Date_ Leg D Condition', 
                                                 'Date_ Leg A Connect Condition', 'Date_ Leg B Connect Condition', 'Date_ Leg C Connect Condition', 'Date_ Leg D Connect Condition']].max(axis=1)

# join leg conditions into one column
df_Found_CA_merged['legs'] = df_Found_CA_merged.apply(lambda row: '-'.join([str(row[col]) for col in ['Measurement_ Leg A Condition', 'Measurement_ Leg B Condition', 'Measurement_ Leg C Condition', 'Measurement_ Leg D Condition'] if pd.notna(row[col])]), axis=1)
df_Found_CA_merged['legs_connect'] = df_Found_CA_merged.apply(lambda row: '-'.join([str(row[col]) for col in ['Measurement_ Leg A Connect Condition', 'Measurement_ Leg B Connect Condition', 'Measurement_ Leg C Connect Condition', 'Measurement_ Leg D Connect Condition'] if pd.notna(row[col])]), axis=1)

# delete columns we don't need anymore
df_Found_CA_merged.drop(columns=['Date_ Leg A Condition', 'Date_ Leg B Condition', 'Date_ Leg C Condition', 'Date_ Leg D Condition', 
                                'Date_ Leg A Connect Condition', 'Date_ Leg B Connect Condition', 'Date_ Leg C Connect Condition', 'Date_ Leg D Connect Condition', 
                                'Measurement_ Leg A Condition', 'Measurement_ Leg B Condition', 'Measurement_ Leg C Condition', 'Measurement_ Leg D Condition',
                                'Measurement_ Leg A Connect Condition', 'Measurement_ Leg B Connect Condition', 'Measurement_ Leg C Connect Condition', 'Measurement_ Leg D Connect Condition'], inplace=True, errors='ignore')


df_Found_CA_merged = df_Found_CA_merged[['structure_id', 'contract', 'type', 'foundation', 'legs', 'legs_connect', 'date']]
df_Found_CA_merged


,structure_id,contract,type,foundation,legs,legs_connect,date
0,OTA-WKM-C0400,C385B,E,PBP,90-90-90-90,95-95-95-95,2023-01-04 09:40:05
1,OTA-WKM-C0401,C385B,C,PBP,70-70-70-70,50-50-50-50,2022-12-02 15:35:19
2,OTA-WKM-C0402,C385B,B,COG,100-100-100-100,100-100-100-100,2024-11-22 00:00:00
3,OTA-WKM-C0403,C385B,C,COG,100-100-100-100,100-100-100-100,2024-11-29 00:00:00
4,OTA-WKM-C0404,C385B,A,GRG,40-40-40-40,20-20-20-20,2022-12-08 09:01:34
...,...,...,...,...,...,...,...
87,OTA-WKM-C0487,C385B,A,COG,75-75-75-75,75-75-75-75,2022-12-05 15:22:41
88,OTA-WKM-C0488,C385B,E,GRG,55-55-55-55,55-50-35-55,2022-12-07 20:27:26
89,OTA-WKM-C0489,C385B,A,GRG,65-65-65-65,65-65-65-65,2022-09-22 07:37:18
90,OTA-WKM-C0490,C385B,A,PBP,85-85-85-85,60-60-20-20,2022-09-15 12:38:41


In [38]:
#df_Found_CA_merged = df_Found_CA_merged.fillna(0)
df_Found_CA_merged['date'] = pd.to_datetime(df_Found_CA_merged['date']) # convert to date format
df_Found_CA_merged['date'] = df_Found_CA_merged['date'].dt.strftime('%Y/%m/%d') # convert to string format for Excel

wb = load_workbook(str_list_xls)

# Remove the existing 'Foundation_CA' sheet if it exists
if 'Foundation_CA' in wb.sheetnames:
    wb.remove(wb['Foundation_CA'])

# Create new 'Foundation_CA' sheet
ws = wb.create_sheet('Foundation_CA')

# Write Foundation_CA dataframe to the sheet
for r in dataframe_to_rows(df_Found_CA_merged, index=False, header=True):
    ws.append(r)

# Save the workbook
wb.save(str_list_xls)

print("Data exported to str_list_xls 'Foundation_CA' sheet.")

Data exported to str_list_xls 'Foundation_CA' sheet.


2.3. Insulators CA  
  
insulators table first

In [40]:
# create a separate df for insulators CA data only to export to Excel
df_ins_CA = str_list_df[[ 'structure_id', 'contract', 'type', 'att_type_phase', 'Jumper Std Assy', 'Strain Back Std Assy', 'Strain Fwd Std Assy', 'Susp Std Assy' ]].copy()    # create a copy of str_list_df to merge with CA data and then export to Excel
df_ins_CA


,structure_id,contract,type,att_type_phase,Jumper Std Assy,Strain Back Std Assy,Strain Fwd Std Assy,Susp Std Assy
0,OTA-WKM-C0400,C385B,E,HSH,CCT2: 10A / CCT1: 848C,CCT2: 13S / CCT1: 760C,CCT2: 13S / CCT1: 760C,None
1,OTA-WKM-C0401,C385B,C,HBK,None,None,None,CCT1: 11CM / CCT2: 11E
2,OTA-WKM-C0402,C385B,B,HSH,None,None,None,CCT2: 10B / CCT1: 10B
3,OTA-WKM-C0403,C385B,C,HBK,None,None,None,CCT2: 10B / CCT1: 10B
4,OTA-WKM-C0404,C385B,A,HSH,None,None,None,CCT1: 11DM / CCT2: 11E
...,...,...,...,...,...,...,...,...
87,OTA-WKM-C0487,C385B,A,PLT,None,None,None,None
88,OTA-WKM-C0488,C385B,E,PLT,CCT2: 523 / CCT1: 523,CCT2: 13S / CCT1: 13S,CCT2: 13S / CCT1: 13S,None
89,OTA-WKM-C0489,C385B,A,HSH,None,None,None,CCT2: 10AS / CCT1: 10AS
90,OTA-WKM-C0490,C385B,A,HSH,None,None,None,CCT1: 524 / CCT2: 524


Sort and rewrite insulators to have them in one column in correct way

In [41]:
# Sort and rewrite insulators to have them in one column in correct way
import re

# function to write numbers only if values are same and sort cct values in correct order if they are different
def normalize_cct(val):
    if pd.isna(val):
        return val

    # extract (CCT#, value)
    pairs = re.findall(r'(CCT\d):\s*([^;]+)', val)

    if len(pairs) != 2:
        return val  # unexpected format

    # convert to dict
    d = {k: v.strip() for k, v in pairs}

    v1 = d.get('CCT1')
    v2 = d.get('CCT2')

    if v1 and v2:
        if v1 == v2:
            return v1  # collapse if same
        else:
            # enforce order: CCT1 first
            return f'CCT1: {v1}; CCT2: {v2}'

    return val


# now we need to join all 4 columns into one
cols = ['Jumper Std Assy', 'Strain Back Std Assy', 'Strain Fwd Std Assy', 'Susp Std Assy']  #insulator columns

def combine_cct(row):
    cct1_vals = []
    cct2_vals = []

    for col in cols:
        val = row[col]
        if pd.isna(val):
            continue

        matches = dict(re.findall(r'(CCT\d):\s*([^/]+)', val))

        if 'CCT1' in matches:
            cct1_vals.append(matches['CCT1'].strip())
        if 'CCT2' in matches:
            cct2_vals.append(matches['CCT2'].strip())

    cct1_str = "/".join(cct1_vals)
    cct2_str = "/".join(cct2_vals)

    return f'CCT1: {cct1_str}; CCT2: {cct2_str}'

df_ins_CA['ins_type'] = df_ins_CA.apply(combine_cct, axis=1)

# apply normalization to insulator CA columns
df_ins_CA['ins_type_norm'] = df_ins_CA['ins_type'].apply(normalize_cct)

# drop original columns we don't need anymore
df_ins_CA.drop(columns=['Jumper Std Assy', 'Strain Back Std Assy', 'Strain Fwd Std Assy', 'Susp Std Assy', 'ins_type'], inplace=True, errors='ignore')
df_ins_CA


,structure_id,contract,type,att_type_phase,ins_type_norm
0,OTA-WKM-C0400,C385B,E,HSH,CCT1: 848C/760C/760C; CCT2: 10A/13S/13S
1,OTA-WKM-C0401,C385B,C,HBK,CCT1: 11CM; CCT2: 11E
2,OTA-WKM-C0402,C385B,B,HSH,10B
3,OTA-WKM-C0403,C385B,C,HBK,10B
4,OTA-WKM-C0404,C385B,A,HSH,CCT1: 11DM; CCT2: 11E
...,...,...,...,...,...
87,OTA-WKM-C0487,C385B,A,PLT,CCT1: ; CCT2:
88,OTA-WKM-C0488,C385B,E,PLT,523/13S/13S
89,OTA-WKM-C0489,C385B,A,HSH,10AS
90,OTA-WKM-C0490,C385B,A,HSH,524


add CA. Here in a different way

In [91]:
# clean up initial CA tab to have insulators and attachments only
# filtering 'Insulator ' and 'Earthwire ' in Asset Description to have only insulators and attachments in CA tab
ca_df_ins = ca_df[ca_df['Asset Description'].str.contains(r'Insulator |Earthwire ', case=False, na=False)].copy()   

# replace "-" symbol in Asset Description to match with structure_id in str_list_df
ca_df_ins['Asset Description'] = ca_df_ins['Asset Description'].str.replace('C-0', 'C0', regex=False)

# replace structure id in Asset Description with nothing to clean up the column
ca_df_ins['Asset Description'] = ca_df_ins.apply(
    lambda row: row['Asset Description'].replace(str(row['structure_id']), '') if pd.notna(row['structure_id']) else row['Asset Description'],
    axis=1)
ca_df_ins

,Asset Description,structure_id,Meter Description,Measurement,Measurement Date
19340,-Insulator Set-OHW-OTA-2,OTA-WKM-C0400,Jumper Cold End Condition,80,2023-01-04 10:23:27
19341,-Insulator Set-OHW-OTA-2,OTA-WKM-C0400,Jumper Hot End Condition,80,2023-01-04 10:23:27
19342,-Insulator Set-OHW-OTA-2,OTA-WKM-C0400,Jumper Ins Condition,80,2023-01-04 10:23:27
19343,-Insulator Set-OHW-OTA-2,OTA-WKM-C0400,Strain Back Cold End Condition,60,2023-01-04 10:23:27
19344,-Insulator Set-OHW-OTA-2,OTA-WKM-C0400,Strain Back Hot End Condition,60,2023-01-04 10:23:27
...,...,...,...,...,...
23912,-Insulator Attachment-OHW-OTA-1,OTA-WKM-C0491,Attachment Point Through Bolt and Nut Condition,70,2022-09-14 14:53:08
23913,-Insulator Attachment-OHW-OTA-2,OTA-WKM-C0491,Attachment Point Condition,65,2022-09-14 14:54:19
23914,-Insulator Attachment-OHW-OTA-2,OTA-WKM-C0491,Attachment Point Nuts and Bolt Condition,60,2022-09-14 14:54:19
23915,-Insulator Attachment-OHW-OTA-2,OTA-WKM-C0491,Attachment Point Steel Condition,75,2022-09-14 14:54:19


In [92]:
# joining Asset Description with Meter Description to have unique values for pivoting

ca_df_ins['Asset Description'] = ca_df_ins['Asset Description'].str.strip() + ' : ' + ca_df_ins['Meter Description'].str.strip()

# drop Meter Description column as it's now joined with Asset Description
ca_df_ins.drop(columns=['Meter Description'], inplace=True, errors='ignore')    

# cleaning text to replace 'Attachments' with 'Att', 'Insulator' with 'Ins', 'Earthwire' with 'EW'
ca_df_ins['Asset Description'] = ca_df_ins['Asset Description'].str.replace('Attachment', 'Att')
ca_df_ins['Asset Description'] = ca_df_ins['Asset Description'].str.replace('Insulator', 'Ins')
ca_df_ins['Asset Description'] = ca_df_ins['Asset Description'].str.replace('Earthwire', 'EW')
ca_df_ins['Asset Description'] = ca_df_ins['Asset Description'].str.replace('Condition', 'CA')

#ca_df_ins

# pivoting table to have one row per structure_id and columns for each asset description with corresponding measurement values
ca_df_ins_T = ca_df_ins.pivot_table(index='structure_id', columns='Asset Description', values=['Measurement', 'Measurement Date'], aggfunc='first').reset_index()
ca_df_ins_T.columns = ['_ '.join([str(i) for i in col if i]) for col in ca_df_ins_T.columns]   # flatten multiindex columns
ca_df_ins_T.columns = [col.replace('Measurement Date_', 'Date_') for col in ca_df_ins_T.columns]     # rename date columns to have Date_ prefix instead of Measurement Date_


#print column names to check
# for col in ca_df_ins_T.columns:
#     print(col)

#ca_df_ins_T

In [93]:
# print to Excel tempopary tab
# replace all NA values with '-' to avoid issues with Excel
ca_df_ins_T = ca_df_ins_T.fillna(999)

wb = load_workbook(str_list_xls)

# Remove the existing 'Ins_CA_temp' sheet if it exists
if 'Ins_CA_temp' in wb.sheetnames:
    wb.remove(wb['Ins_CA_temp'])

# Create new 'Ins_CA_temp' sheet
ws = wb.create_sheet('Ins_CA_temp')

# Write Ins_CA_temp dataframe to the sheet
for r in dataframe_to_rows(ca_df_ins_T, index=False, header=True):
    ws.append(r)

# Save the workbook
wb.save(str_list_xls)

print("Data exported to str_list_xls 'Ins_CA_temp' sheet.")


Data exported to str_list_xls 'Ins_CA_temp' sheet.


In [94]:
#convert all date columns to date format for further processing
for col in ca_df_ins_T.columns:
    if col.startswith('Date_'):
        ca_df_ins_T[col] = pd.to_datetime(ca_df_ins_T[col], errors='coerce')        


# joint columns with minimum values and most recent dates for insulator CA data
# EW Att
ca_df_ins_T['EW_att'] = ca_df_ins_T.filter(like='Measurement_ -EW Att').min(axis=1)
ca_df_ins_T['EW_Att_date'] = ca_df_ins_T.filter(like='Date_ -EW Att').max(axis=1)

# phase attachments
ca_df_ins_T['phase_att'] = ca_df_ins_T.filter(like='Measurement_ -Ins Att').min(axis=1)
ca_df_ins_T['phase_att_date'] = ca_df_ins_T.filter(like='Date_ -Ins Att').max(axis=1)


# use constrains for other columns 
# phase insulators
cols_phase_m = ca_df_ins_T.columns[ca_df_ins_T.columns.str.contains(r'Measurement_\s*-Ins Set.*:\s*S')]
cols_phase_d = ca_df_ins_T.columns[ca_df_ins_T.columns.str.contains(r'Date_\s*-Ins Set.*:\s*S')]

ca_df_ins_T['phase_ins'] = ca_df_ins_T[cols_phase_m].min(axis=1)
ca_df_ins_T['phase_ins_date'] = ca_df_ins_T[cols_phase_d].max(axis=1)

# jumpers
cols_jum_m = ca_df_ins_T.columns[ca_df_ins_T.columns.str.contains(r'Measurement_\s*-Ins Set.*:\s*Jumper')]
cols_jum_d = ca_df_ins_T.columns[ca_df_ins_T.columns.str.contains(r'Date_\s*-Ins Set.*:\s*Jumper')]

ca_df_ins_T['jumper_ins'] = ca_df_ins_T[cols_jum_m].min(axis=1)
ca_df_ins_T['jumper_ins_date'] = ca_df_ins_T[cols_jum_d].max(axis=1)

# armour rod
cols_ar_m = ca_df_ins_T.columns[ca_df_ins_T.columns.str.contains(r'Measurement_\s*-Ins Set.*:\s*Armour')]
cols_ar_d = ca_df_ins_T.columns[ca_df_ins_T.columns.str.contains(r'Date_\s*-Ins Set.*:\s*Armour')]

ca_df_ins_T['AR_ins'] = ca_df_ins_T[cols_ar_m].min(axis=1)
ca_df_ins_T['AR_ins_date'] = ca_df_ins_T[cols_ar_d].max(axis=1)

# weights
cols_wt_m = ca_df_ins_T.columns[ca_df_ins_T.columns.str.contains(r'Measurement_\s*-Ins Set.*:\s*Weight')]
cols_wt_d = ca_df_ins_T.columns[ca_df_ins_T.columns.str.contains(r'Date_\s*-Ins Set.*:\s*Weight')]

ca_df_ins_T['WT_ins'] = ca_df_ins_T[cols_wt_m].min(axis=1)
ca_df_ins_T['WT_ins_date'] = ca_df_ins_T[cols_wt_d].max(axis=1)

# rename columns to have more user-friendly names
ca_df_ins_T.rename(columns={
    'Measurement_ -EW HW Set : Armour Rod CA': 'EW_AR',
    'Date_ -EW HW Set : Armour Rod CA': 'EW_AR_date',
    'Measurement_ -EW HW Set : EW Hardware CA' : 'EW_HW',
    'Date_ -EW HW Set : EW Hardware CA' : 'EW_HW_date'
}, inplace=True)

ca_df_ins_T_final = ca_df_ins_T[['structure_id', 'EW_att', 'EW_Att_date', 'EW_AR', 'EW_AR_date', 'EW_HW', 'EW_HW_date', 'phase_att', 'phase_att_date', 'phase_ins', 'phase_ins_date', 'jumper_ins', 'jumper_ins_date', 'AR_ins', 'AR_ins_date', 'WT_ins', 'WT_ins_date']]
ca_df_ins_T_final


,structure_id,EW_att,EW_Att_date,EW_AR,EW_AR_date,EW_HW,EW_HW_date,phase_att,phase_att_date,phase_ins,phase_ins_date,jumper_ins,jumper_ins_date,AR_ins,AR_ins_date,WT_ins,WT_ins_date
0,OTA-WKM-C0400,60,2023-01-04 10:23:57,60,2023-01-04 10:34:55,75,2023-01-04 10:34:55,95,2023-01-04 09:52:21,50,2023-01-04 10:23:27,80,2023-01-04 10:23:27,999,1970-01-01 00:00:00.000000999,999,1970-01-01 00:00:00.000000999
1,OTA-WKM-C0401,50,2022-12-02 15:42:24,40,2022-12-02 16:42:05,70,2022-12-02 16:42:05,40,2024-10-30 00:00:00,100,2024-10-30 00:00:00,999,NaT,100,2024-10-30 00:00:00.000000000,999,1970-01-01 00:00:00.000000999
2,OTA-WKM-C0402,100,2023-05-18 13:11:30,50,2023-05-18 11:27:22,100,2023-05-18 11:27:01,40,2022-12-08 13:47:21,30,2022-12-08 13:46:06,999,NaT,70,2022-12-08 13:46:06.000000000,999,1970-01-01 00:00:00.000000999
3,OTA-WKM-C0403,100,2023-05-11 12:25:05,45,2023-05-11 12:28:14,100,2023-05-11 12:28:01,40,2022-12-08 12:01:09,30,2022-12-08 11:58:14,999,NaT,70,2022-12-08 11:58:14.000000000,999,1970-01-01 00:00:00.000000999
4,OTA-WKM-C0404,100,2023-05-30 00:00:00,100,2023-05-30 07:52:15,100,2023-05-30 07:53:04,45,2022-12-08 09:30:57,100,2025-05-08 00:00:00,999,NaT,100,2025-05-08 00:00:00.000000000,999,1970-01-01 00:00:00.000000999
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87,OTA-WKM-C0487,75,2022-12-05 15:35:57,35,2022-12-05 15:34:08,75,2022-12-05 15:34:08,75,2022-12-05 15:57:18,75,2022-12-05 15:56:18,999,NaT,75,2022-12-05 15:56:18.000000000,999,1970-01-01 00:00:00.000000999
88,OTA-WKM-C0488,60,2022-12-07 19:24:07,999,NaT,55,2022-12-07 19:23:06,55,2024-09-23 19:35:42,55,2022-12-07 20:01:54,45,2022-12-07 20:01:54,999,1970-01-01 00:00:00.000000999,999,1970-01-01 00:00:00.000000999
89,OTA-WKM-C0489,70,2022-09-22 07:40:12,35,2022-09-22 07:42:41,70,2022-09-22 07:42:41,50,2022-09-22 07:41:17,45,2022-09-22 07:39:16,999,NaT,85,2022-09-22 07:39:16.000000000,999,1970-01-01 00:00:00.000000999
90,OTA-WKM-C0490,50,2022-09-15 14:58:05,35,2022-09-15 15:09:05,50,2022-09-15 15:09:05,55,2022-09-15 15:03:30,80,2022-09-15 14:51:09,999,NaT,85,2022-09-15 14:51:09.000000000,999,1970-01-01 00:00:00.000000999


Merge 

In [95]:
# merge with df_ins_CA
df_ins_CA_final = df_ins_CA.merge(ca_df_ins_T_final, on='structure_id', how='left')
#df_ins_CA_final


In [96]:
# print to Excel tempopary tab
# replace all NA values with '-' to avoid issues with Excel
df_ins_CA_final = df_ins_CA_final.fillna(0)

wb = load_workbook(str_list_xls)

# Remove the existing 'Ins_CA' sheet if it exists
if 'Ins_CA' in wb.sheetnames:
    wb.remove(wb['Ins_CA'])

# Create new 'Ins_CA' sheet
ws = wb.create_sheet('Ins_CA')

# Write Ins_CA dataframe to the sheet
for r in dataframe_to_rows(df_ins_CA_final, index=False, header=True):
    ws.append(r)

# Save the workbook
wb.save(str_list_xls)

print("Data exported to str_list_xls 'Ins_CA' sheet.")


Data exported to str_list_xls 'Ins_CA' sheet.
